In [1]:
from IPython.display import display, HTML 
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [3]:
# 라이브러리 업로드
import requests
import json
import time
import pandas as pd
from datetime import datetime
import re
import logging
from typing import List, Dict, Optional
from urllib.parse import urlparse
from collections import defaultdict

In [33]:
class PeriodEstimator:
    """업로드 시기 추정을 위한 클래스"""
    
    def __init__(self):
        # 도메인별 시기 추정 가중치 (실제 플랫폼 특성 반영)
        self.domain_weights = {
            # 2024년 6월에 더 가까운 도메인들 (전통적 플랫폼)
            'older_domains': {
                'blog.naver.com': 0.8,
                'cafe.naver.com': 0.8,
                'tistory.com': 0.7,
                'egloos.com': 0.9,
                'wordpress.com': 0.6,
                'blogger.com': 0.7,
                'daum.net': 0.7,
                'daumcafe.net': 0.8,
            },
            # 2025년 6월에 더 가까운 도메인들 (최신 플랫폼)
            'newer_domains': {
                'instagram.com': 0.9,
                'facebook.com': 0.8,
                'twitter.com': 0.9,
                'x.com': 0.9,
                'youtube.com': 0.8,
                'youtu.be': 0.8,
                'tiktok.com': 0.95,
                'threads.net': 0.95,
                'pinterest.com': 0.7,
                'baemin.com': 0.9,
                'yogiyo.co.kr': 0.9,
                'coupang.com': 0.8,
                'kurly.com': 0.8,
            }
        }

    def extract_domain(self, url: str) -> str:
        """URL에서 도메인 추출"""
        try:
            parsed = urlparse(url.lower())
            domain = parsed.netloc
            if domain.startswith('www.'):
                domain = domain[4:]
            return domain
        except:
            return "unknown"

    def analyze_url_date_patterns(self, url: str) -> tuple[Optional[str], float]:
        """URL에서 날짜 패턴 분석"""
        url_lower = url.lower()
        
        # 확실한 2024년 6월 패턴
        patterns_2024_06 = [
            r'/2024/0?6/',
            r'/202406/',
            r'2024-0?6-\d',
            r'date[=:]2024[/-]0?6',
        ]
        
        for pattern in patterns_2024_06:
            if re.search(pattern, url_lower):
                return "2024-06", 1.0
        
        # 확실한 2025년 6월 패턴
        patterns_2025_06 = [
            r'/2025/0?6/',
            r'/202506/',
            r'2025-0?6-\d',
            r'date[=:]2025[/-]0?6',
        ]
        
        for pattern in patterns_2025_06:
            if re.search(pattern, url_lower):
                return "2025-06", 1.0
        
        # 일반적인 2024년 패턴 (6월 근처)
        if re.search(r'/2024/0?[4-8]/', url_lower) or re.search(r'2024-0?[4-8]-', url_lower):
            return "2024-06", 0.6
        
        # 일반적인 2025년 패턴 (6월 근처)
        if re.search(r'/2025/0?[4-8]/', url_lower) or re.search(r'2025-0?[4-8]-', url_lower):
            return "2025-06", 0.6
        
        return None, 0.0

    def analyze_domain_characteristics(self, domain: str) -> tuple[Optional[str], float]:
        """도메인 특성 분석"""
        domain_lower = domain.lower()
        
        # 오래된 플랫폼 체크
        for old_domain, weight in self.domain_weights['older_domains'].items():
            if old_domain in domain_lower:
                return "2024-06", weight
        
        # 새로운 플랫폼 체크
        for new_domain, weight in self.domain_weights['newer_domains'].items():
            if new_domain in domain_lower:
                return "2025-06", weight
        
        return None, 0.0

    def estimate_upload_period(self, url: str) -> str:
        """종합적인 업로드 시기 추정"""
        if not url:
            return "2025-06"
        
        domain = self.extract_domain(url)
        
        # 1. URL 날짜 패턴 분석 (최우선)
        period, confidence = self.analyze_url_date_patterns(url)
        if period and confidence >= 0.6:
            return period
        
        # 2. 도메인 특성 분석
        period, confidence = self.analyze_domain_characteristics(domain)
        if period and confidence >= 0.7:
            return period
        
        # 3. 기본값 (최신으로 추정)
        return "2025-06"

In [30]:
class NaverFoodImageScraper:
    def __init__(self, client_id: str, client_secret: str):
        self.client_id = client_id
        self.client_secret = client_secret
        self.base_url = "https://openapi.naver.com/v1/search"
        self.headers = {
            'X-Naver-Client-Id': client_id,
            'X-Naver-Client-Secret': client_secret,
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        # 시기 추정기 초기화
        self.period_estimator = PeriodEstimator()

        # 단품 음식 키워드 (실제 검색량 기반 정렬)
        self.food_keywords = [
            '치킨', '피자', '햄버거', '라면', '짜장면', '짬뽕', '김치찌개', '된장찌개',
            '불고기', '갈비', '삼겹살', '비빔밥', '냉면', '순두부찌개', '김밥', '떡볶이',
            '순대', '호떡', '붕어빵', '초밥', '회', '삼계탕', '설렁탕', '곰탕',
            '갈비탕', '육개장', '김치', '계란말이', '부침개', '만두', '족발', '보쌈',
            '닭갈비', '스테이크', '파스타', '타코야키', '연어', '새우', '게', '랍스터'
        ]

        # 키워드별 실제 검색량 추정치
        self.keyword_popularity = {
            '치킨': 9500, '피자': 8200, '햄버거': 7800, '라면': 9000, '짜장면': 6500,
            '짬뽕': 6200, '김치찌개': 5800, '된장찌개': 4200, '불고기': 5500, '갈비': 6800,
            '삼겹살': 7200, '비빔밥': 4800, '냉면': 5200, '순두부찌개': 3800, '김밥': 5500,
            '떡볶이': 6200, '순대': 3500, '호떡': 2800, '붕어빵': 2200, '초밥': 4500,
            '회': 4200, '삼계탕': 3800, '설렁탕': 3200, '곰탕': 2800, '갈비탕': 3500,
            '육개장': 3200, '김치': 4800, '계란말이': 3200, '부침개': 2800, '만두': 4200,
            '족발': 3800, '보쌈': 3500, '닭갈비': 3200, '스테이크': 5500, '파스타': 5800,
            '타코야키': 2500, '연어': 4200, '새우': 4500, '게': 3800, '랍스터': 2200
        }
        
class NaverFoodImageScraper:
    def __init__(self, client_id: str, client_secret: str):
        self.client_id = client_id
        self.client_secret = client_secret
        self.base_url = "https://openapi.naver.com/v1/search"
        self.headers = {
            'X-Naver-Client-Id': client_id,
            'X-Naver-Client-Secret': client_secret,
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        # 시기 추정기 초기화
        self.period_estimator = PeriodEstimator()

        # 단품 음식 키워드 (실제 검색량 기반 정렬)
        self.food_keywords = [
            '치킨', '피자', '햄버거', '라면', '짜장면', '짬뽕', '김치찌개', '된장찌개',
            '불고기', '갈비', '삼겹살', '비빔밥', '냉면', '순두부찌개', '김밥', '떡볶이',
            '순대', '호떡', '붕어빵', '초밥', '회', '삼계탕', '설렁탕', '곰탕',
            '갈비탕', '육개장', '김치', '계란말이', '부침개', '만두', '족발', '보쌈',
            '닭갈비', '스테이크', '파스타', '타코야키', '연어', '새우', '게', '랍스터'
        ]

        # 키워드별 실제 검색량 추정치
        self.keyword_popularity = {
            '치킨': 9500, '피자': 8200, '햄버거': 7800, '라면': 9000, '짜장면': 6500,
            '짬뽕': 6200, '김치찌개': 5800, '된장찌개': 4200, '불고기': 5500, '갈비': 6800,
            '삼겹살': 7200, '비빔밥': 4800, '냉면': 5200, '순두부찌개': 3800, '김밥': 5500,
            '떡볶이': 6200, '순대': 3500, '호떡': 2800, '붕어빵': 2200, '초밥': 4500,
            '회': 4200, '삼계탕': 3800, '설렁탕': 3200, '곰탕': 2800, '갈비탕': 3500,
            '육개장': 3200, '김치': 4800, '계란말이': 3200, '부침개': 2800, '만두': 4200,
            '족발': 3800, '보쌈': 3500, '닭갈비': 3200, '스테이크': 5500, '파스타': 5800,
            '타코야키': 2500, '연어': 4200, '새우': 4500, '게': 3800, '랍스터': 2200
        }
        
    def search_images(self, query: str, display: int = 100, start: int = 1, sort: str = 'sim') -> Optional[Dict]:
        """네이버 이미지 검색 API 호출"""
        url = f"{self.base_url}/image"
        params = {
            'query': query,
            'display': display,
            'start': start,
            'sort': sort
        }
        
        try:
            response = requests.get(url, headers=self.headers, params=params, timeout=10)
            response.raise_for_status()
            
            result = response.json()
            if 'items' not in result:
                logger.warning(f"검색 결과 없음: {query}")
                return None
                
            logger.debug(f"검색 성공: {query} - {len(result['items'])}개 결과")
            return result
            
        except requests.exceptions.HTTPError as e:
            if hasattr(response, 'status_code'):
                if response.status_code == 400:
                    logger.error(f"잘못된 요청 ({query}): API 키 또는 파라미터 확인 !")
                elif response.status_code == 403:
                    logger.error(f"권한 없음 ({query}): API 키 권한 확인 필요 !")
                elif response.status_code == 429:
                    logger.error(f"요청 한도 초과 ({query}): 잠시 대기 후 재시도 !")
                    time.sleep(5)
                else:
                    logger.error(f"HTTP 오류 ({query}): {e} !")
            else:
                logger.error(f"HTTP 오류 ({query}): {e} !")
            return None
            
        except requests.exceptions.RequestException as e:
            logger.error(f"네트워크 오류 ({query}): {e} !")
            return None
        except json.JSONDecodeError as e:
            logger.error(f"JSON 파싱 오류 ({query}): {e} !")
            return None
        
    def get_web_search_volume(self, keyword: str) -> int:
        """웹검색 결과 수로 실제 검색량 추정"""
        try:
            url = f"{self.base_url}/webkr"
            params = {
                'query': f"{keyword} 음식",
                'display': 1
            }
            
            response = requests.get(url, headers=self.headers, params=params, timeout=10)
            if response.status_code == 200:
                result = response.json()
                total_count = result.get('total', 0)
                # 결과 수를 검색량으로 변환
                estimated_volume = min(int(total_count / 1000) + 1000, 10000)
                logger.debug(f"{keyword} 웹검색: {total_count:,}개 -> 추정량: {estimated_volume}")
                return estimated_volume
            else:
                return self.keyword_popularity.get(keyword, 1000)
                
        except Exception as e:
            logger.debug(f"웹검색량 추정 실패 ({keyword}): {e} !")
            return self.keyword_popularity.get(keyword, 1000)
    
    def extract_image_info(self, item: Dict, keyword: str, search_volume: int) -> Dict:
        """이미지 정보 추출 및 시기 추정"""
        title = item.get('title', '').replace('<b>', '').replace('</b>', '').strip()
        link = item.get('link', '')
        thumbnail = item.get('thumbnail', '')
        domain = self.period_estimator.extract_domain(link)
        estimated_period = self.period_estimator.estimate_upload_period(link)
        
        return {
            'keyword': keyword,
            'search_volume': search_volume,
            'title': title,
            'image_url': link,
            'thumbnail_url': thumbnail,
            'domain': domain,
            'estimated_period': estimated_period,
            'collected_at': datetime.now().isoformat()
        }

    def scrape_food_images_by_popularity(self, target_periods: List[str], max_results_per_period: int = 500) -> pd.DataFrame:
        """검색량 기반 음식 이미지 수집"""
        all_results = []
        
        # 실제 검색량 조사 (상위 키워드만)
        logger.info("키워드별 검색량 조사 중...")
        keyword_volumes = {}
        
        # 상위 50개(최대 수집 수의 1/10)만 실제 조사 (API 호출 최소화)
        no_keywords = int(max_results_per_period/10)
        for i, keyword in enumerate(self.food_keywords[:no_keywords]):
            volume = self.get_web_search_volume(keyword)
            keyword_volumes[keyword] = volume
            logger.info(f"[{i+1}/{no_keywords}] {keyword}: {volume:,}")
            time.sleep(0.5)
        
        # 나머지는 기본값 사용
        for keyword in self.food_keywords[no_keywords:]:
            keyword_volumes[keyword] = self.keyword_popularity.get(keyword, 1000)
        
        # 검색량 순으로 정렬
        sorted_keywords = sorted(keyword_volumes.items(), key=lambda x: x[1], reverse=True)
        
        # 기간별 수집 현황 추적
        collected_by_period = {period: 0 for period in target_periods}
        total_target = len(target_periods) * max_results_per_period
        
        logger.info(f"데이터 수집 시작 - 목표: 각 기간 {max_results_per_period}개씩, 총 {total_target}개")
        
        for keyword, search_volume in sorted_keywords:
            # 모든 기간이 목표에 도달하면 종료
            if all(count >= max_results_per_period for count in collected_by_period.values()):
                logger.info("모든 기간의 목표 수집량 달성!")
                break
                
            logger.info(f"수집 중: {keyword} (검색량: {search_volume:,})")
            keyword_results = []

            # 최대 1000개까지 검색 (API 제한)
            for start_pos in range(1, 901, 100):  # 안전하게 900까지만
                search_result = self.search_images(
                    query=f"{keyword} 음식 요리",
                    display=100,
                    start=start_pos,
                    sort='sim'
                )
                
                if not search_result or 'items' not in search_result:
                    break
                    
                items = search_result['items']
                if not items:
                    break
                
                for item in items:
                    image_info = self.extract_image_info(item, keyword, search_volume)
                    keyword_results.append(image_info)
                
                time.sleep(0.1)
                
                # 키워드당 최대 150개로 제한 (효율성)
                if len(keyword_results) >= 150:
                    break
            
            # 기간별로 분류하여 추가
            period_filtered = defaultdict(list)
            for result in keyword_results:
                period = result['estimated_period']
                if period in target_periods:
                    period_filtered[period].append(result)
            
            # 각 기간별로 균등하게 분배
            added_count = 0
            for period, results in period_filtered.items():
                if collected_by_period[period] < max_results_per_period:
                    needed = max_results_per_period - collected_by_period[period]
                    to_add = results[:needed]
                    all_results.extend(to_add)
                    collected_by_period[period] += len(to_add)
                    added_count += len(to_add)
            
            logger.info(f"{keyword}: {added_count}개 추가, 진행률: {collected_by_period}")
            time.sleep(1)

        # DataFrame 생성 및 정리
        df = pd.DataFrame(all_results)
        if not df.empty:
            # 중복 제거 (URL 기준)
            df = df.drop_duplicates(subset=['image_url'], keep='first')
            # 검색량 기준 정렬
            df = df.sort_values(['search_volume', 'keyword'], ascending=[False, True]).reset_index(drop=True)

        return df
    
    def get_collection_stats(self, df: pd.DataFrame) -> Dict:
        """수집 통계 생성"""
        if df.empty:
            return {"error": "수집된 데이터가 없습니다"}
        
        stats = {
            'summary': {
                'total_images': len(df),
                'unique_keywords': df['keyword'].nunique(),
                'unique_domains': df['domain'].nunique(),
                'collection_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            },
            'period_distribution': df['estimated_period'].value_counts().to_dict(),
            'keyword_ranking': df.groupby('keyword').agg({
                'search_volume': 'first',
                'image_url': 'count'
            }).sort_values('search_volume', ascending=False).head(10).to_dict(),
            'domain_distribution': df['domain'].value_counts().head(10).to_dict(),
            'search_volume_stats': {
                'mean': float(df['search_volume'].mean()),
                'median': float(df['search_volume'].median()),
                'max': int(df['search_volume'].max()),
                'min': int(df['search_volume'].min())
            }
        }
        
        return stats

In [31]:
def run_scraper(client_id: str, client_secret: str) -> tuple[pd.DataFrame, Dict]:
    """스크래퍼 실행 및 결과 반환"""
    scraper = NaverFoodImageScraper(client_id, client_secret)
    target_periods = ["2024-06", "2025-06"]
    
    # 데이터 수집
    df = scraper.scrape_food_images_by_popularity(target_periods, max_results_per_period=500)
    
    # 통계 생성
    stats = scraper.get_collection_stats(df)
    
    return df, stats

In [34]:
# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def main():
    """메인 실행 함수"""
    # 네이버 API 키 로드
    load_dotenv()
    client_id = os.getenv('Client_ID')
    client_secret = os.getenv('Client_Secret')

    if not client_id:
        print("네이버 API 키를 확인!")
        return 
    
    try:
        # 스크래퍼 실행
        result_df, stats = run_scraper(client_id, client_secret)

        if not result_df.empty:
            # 결과 저장
            filename = f"naver_food_images_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            result_df.to_csv(filename, index=False, encoding='utf-8-sig')
            
            # 결과 출력
            print(f"\n수집 완료!")
            print(f"총 이미지: {stats['summary']['total_images']:,}개")
            print(f"고유 키워드: {stats['summary']['unique_keywords']}개")
            print(f"고유 도메인: {stats['summary']['unique_domains']}개")
            print(f"저장 파일: {filename}")
            
            # 기간별 분포
            print(f"\n기간별 분포:")
            for period, count in stats['period_distribution'].items():
                print(f"  {period}: {count:,}개")
            
            # 상위 키워드
            print(f"\n상위 키워드 (검색량 기준):")
            for i, (keyword, data) in enumerate(list(stats['keyword_ranking']['search_volume'].items())[:5], 1):
                volume = data
                count = stats['keyword_ranking']['image_url'][keyword]
                print(f"  {i}. {keyword}: 검색량 {volume:,}, 이미지 {count}개")
            
            # 주요 도메인
            print(f"\n주요 도메인:")
            for i, (domain, count) in enumerate(list(stats['domain_distribution'].items())[:5], 1):
                print(f"  {i}. {domain}: {count}개")
            
            # 샘플 데이터
            print(f"\n샘플 데이터:")
            sample_cols = ['keyword', 'search_volume', 'title', 'estimated_period']
            print(result_df[sample_cols].head(3).to_string(index=False))
            
        else:
            print("데이터 수집 실패 !")
        
    except Exception as e:
        logger.error(f"실행 중 오류 : {e} !")
        print(f"오류 발생: {e} !")
        
if __name__ == "__main__":
    main()

2025-07-14 18:09:52,268 - INFO - 키워드별 검색량 조사 중...
2025-07-14 18:09:52,620 - INFO - [1/50] 치킨: 10,000
2025-07-14 18:09:53,406 - INFO - [2/50] 피자: 10,000
2025-07-14 18:09:54,164 - INFO - [3/50] 햄버거: 5,977
2025-07-14 18:09:54,981 - INFO - [4/50] 라면: 10,000
2025-07-14 18:09:55,836 - INFO - [5/50] 짜장면: 5,621
2025-07-14 18:09:56,624 - INFO - [6/50] 짬뽕: 6,587
2025-07-14 18:09:57,405 - INFO - [7/50] 김치찌개: 6,115
2025-07-14 18:09:58,182 - INFO - [8/50] 된장찌개: 3,733
2025-07-14 18:09:58,995 - INFO - [9/50] 불고기: 5,850
2025-07-14 18:09:59,768 - INFO - [10/50] 갈비: 10,000
2025-07-14 18:10:00,551 - INFO - [11/50] 삼겹살: 8,367
2025-07-14 18:10:01,308 - INFO - [12/50] 비빔밥: 6,085
2025-07-14 18:10:02,073 - INFO - [13/50] 냉면: 6,688
2025-07-14 18:10:02,831 - INFO - [14/50] 순두부찌개: 2,396
2025-07-14 18:10:03,615 - INFO - [15/50] 김밥: 10,000
2025-07-14 18:10:04,371 - INFO - [16/50] 떡볶이: 8,161
2025-07-14 18:10:05,125 - INFO - [17/50] 순대: 5,771
2025-07-14 18:10:05,875 - INFO - [18/50] 호떡: 2,171
2025-07-14 18:10:06,626


수집 완료!
총 이미지: 563개
고유 키워드: 32개
고유 도메인: 115개
저장 파일: naver_food_images_20250714_181136.csv

기간별 분포:
  2025-06: 498개
  2024-06: 65개

상위 키워드 (검색량 기준):
  1. 갈비: 검색량 10,000, 이미지 1개
  2. 라면: 검색량 10,000, 이미지 104개
  3. 김밥: 검색량 10,000, 이미지 3개
  4. 김치: 검색량 10,000, 이미지 5개
  5. 피자: 검색량 10,000, 이미지 199개

주요 도메인:
  1. i.pinimg.com: 98개
  2. imgnews.naver.net: 96개
  3. cdn.crowdpic.net: 47개
  4. sk5.co.kr: 39개
  5. recipe1.ezmember.co.kr: 29개

샘플 데이터:
keyword  search_volume                                                                            title estimated_period
     갈비          10000 한우 요리 :: 한우 요리,소갈비찜,치즈 고추장 한우갈비,한우,한우요리 :: 맛집 정보 검색 NO.1 사이트, 메뉴판닷컴 :: 맛집 정보가 가득          2024-06
     김밥          10000  매생이로 만들 수 있는 음식레시피 좀 가르.. :: 매생이,뚜뚜루,매생이 계란말이,매생이 두부 달걀 밥,매생이바지락국,매생이요리,매생이전...          2024-06
     김밥          10000                        웹진 인벤 : 김밥이랑 제일 잘 어울리는 음식은? - 오픈이슈갤러리 김밥이랑 제일 잘 어울리는 음식은?          2024-06
